# Streaming Graph Updates

This notebook illustrates how to use `update()` to incrementally update embeddings as the graph evolves, without a full refit from scratch.

> **Note:** The current implementation performs a warm-started refit. Woodbury-based $O(N^2)$ incremental updates for the forest matrix are planned for a future release.

In [ ]:
import networkx as nx
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from hypegrl.embedders.poincare_maps import PoincareMapsEmbedder
from hypegrl.visualization.disk import plot_poincare_graph

## Initial fit on a balanced tree

In [ ]:
G0 = nx.balanced_tree(r=2, h=3)
print(f'Initial graph: {G0.number_of_nodes()} nodes, {G0.number_of_edges()} edges')

embedder = PoincareMapsEmbedder(
    d=2, n_steps=200, log_every=0,
    regularize_a=0.01, random_state=0,
)
embedder.fit(G0)
print(f'Fit complete. Final loss: {embedder.loss_history[-1]:.4f}')

## Add new edges (streaming update)

New edges arrive and are initially treated as unknown.

In [ ]:
# Two new edges arrive
new_edges = [(0, 14), (3, 12)]
embedder.update(added_edges=new_edges)
print(f'After edge addition. Loss: {embedder.loss_history[-1]:.4f}')
print(f'Unknown edges (newly added): {embedder._unknown_edges}')

## Reveal edge weights

After observation, the true weights become known.

In [ ]:
embedder.update(
    revealed_edges={(0, 14): 1.0, (3, 12): 1.0}
)
print(f'After revealing edges. Loss: {embedder.loss_history[-1]:.4f}')
print(f'Unknown edges remaining: {embedder._unknown_edges}')

## Remove an edge

Note: removal raises an error if it disconnects the graph.

In [ ]:
# Removing a non-bridge edge from the augmented graph
try:
    embedder.update(removed_edges=[(0, 14)])
    print('Edge removed successfully')
except ValueError as e:
    print(f'Caught expected error: {e}')

## Visualise final embeddings

In [ ]:
fig = plot_poincare_graph(
    embedder._G, embedder.embeddings(),
    title='Balanced tree after streaming updates',
)
fig.savefig('streaming_result.png', dpi=150, bbox_inches='tight')